In [6]:
from pathlib import Path
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import gcamreader

In [7]:
# =============================
# Config
# =============================
PROJECT_PATH   = Path("/data/project/tae/gcam-core")
DB_REL_PATH    = "../output"   # relative to current working directory
DB_FILE        = "database_basexdb_korea_2035_v7"
QUERY_FILE     = Path("..") / "output" / "queries" / "Main_queries.xml"

REGION         = "South Korea"
SCENARIOS      = ["Current-Policies-Med", "High-Ambition-Med"]
YEARS_TICKS    = list(range(2005, 2036, 5))

# Unit conversion: 1 EJ = 23.8846 Mtoe
EJ_TO_MTOE     = 23.8846
Y_AXIS_TITLE   = "Mtoe"

# Fuel display order & styling
STACK_ORDER = [
    "Electricity", "Biomass", "Traditional Biomass", "Gas", "Coal", "Oil", "Hydrogen"
]

FUEL_MAP = {
    "delivered gas"            : "Gas",
    "elect_td_bld"             : "Electricity",
    "H2 retail delivery"       : "Hydrogen",
    "delivered biomass"        : "Biomass",
    "refined liquids enduse"   : "Oil",
    "delivered coal"           : "Coal",
    "traditional biomass"      : "Traditional Biomass",
}

FUEL_COLORS = {
    "Traditional Biomass": "#FFB785",  # peach
    "Oil"               : "#EF0C0C",   # vivid red
    "Coal"              : "#000000",   # black
    "Gas"               : "#0186E0",   # bright blue
    "Biomass"           : "#099B43",   # medium green
    "Electricity"       : "#0057B5",   # deep blue
    "Hydrogen"          : "#FED30B",   # golden yellow
}


In [8]:
# =============================
# Helpers
# =============================
def categorize_fuel(series: pd.Series) -> pd.Series:
    """Map GCAM input names to display fuel categories."""
    out = series.map(FUEL_MAP)
    # Keep any unknowns visible (rather than dropping them silently)
    return out.fillna(series)

def prep_data(conn, query_idx: int) -> pd.DataFrame:
    """Run a single query, convert units, categorize fuels, and aggregate."""
    queries   = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q         = queries[query_idx]
    df        = conn.runQuery(q, scenarios=SCENARIOS, regions=[REGION])

    # Use first token of scenario name (often '<name>, <suffix>')
    df["scenario"] = df["scenario"].str.split(",").str[0]

    # Convert EJ -> Mtoe (axis label set accordingly)
    df["value"] = df["value"] * EJ_TO_MTOE

    # Fuel categorization & ordering
    df["fuel"] = categorize_fuel(df["input"])
    df["fuel"] = pd.Categorical(df["fuel"], categories=STACK_ORDER, ordered=True)

    # Aggregate
    df = (
        df[df["Year"] >= 2005]
        .groupby(["scenario", "Year", "fuel"], as_index=False)["value"]
        .sum()
        .sort_values(["scenario", "Year", "fuel"])
    )
    return df, q.title

def add_stacked_bars(fig, data: pd.DataFrame, col: int, show_legend: bool = True):
    """Add stacked bars for one scenario pane."""
    fuels = [f for f in STACK_ORDER if f in data["fuel"].unique()]
    for fuel in fuels:
        sub = data[data["fuel"] == fuel]
        fig.add_bar(
            name=fuel,
            x=sub["Year"],
            y=sub["value"],
            marker=dict(color=FUEL_COLORS.get(fuel)),
            row=1, col=col,
            showlegend=show_legend
        )

In [9]:
# Connect DB
conn = gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

# Choose the query index you need (was 77 in your original)
QUERY_INDEX = 77

df, query_title = prep_data(conn, QUERY_INDEX)

# Split per scenario
df_cp = df[df["scenario"] == SCENARIOS[0]]
df_ep = df[df["scenario"] == SCENARIOS[1]]

Database scenarios: High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, Current-Policies-Med, High-Ambition-High, High-Ambition-Low, High-Ambition-Med-AI, High-Ambition-Med-CPO2040, Current-Policies-Low, Current-Policies-High, Current-Policies-Med-AI


/tmp/ipykernel_151566/2381546345.py:28: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [14]:
df_cp[(df_cp['fuel']=='Electricity')].groupby(['Year'])['value'].sum() / df_cp.groupby(['Year'])['value'].sum()

Year
2005    0.366536
2010    0.446788
2015    0.462684
2020    0.471611
2025    0.514780
2030    0.550436
2035    0.567762
Name: value, dtype: float64

In [13]:
df_ep[(df_ep['fuel']=='Electricity')].groupby(['Year'])['value'].sum() / df_ep.groupby(['Year'])['value'].sum()

Year
2005    0.366536
2010    0.446788
2015    0.462684
2020    0.471611
2025    0.514780
2030    0.586550
2035    0.686311
Name: value, dtype: float64

In [10]:
# Figure with two panes
fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True, shared_xaxes=True,
    specs=[[{"secondary_y": False}, {"secondary_y": False}]],
    subplot_titles=("Current Policies", "High Ambition")
)

add_stacked_bars(fig, df_cp, col=1)
add_stacked_bars(fig, df_ep, col=2, show_legend=False)

# Layout (single pass)
fig.update_layout(
    barmode="stack",
    plot_bgcolor="rgba(0,0,0,0)",
    width=900, height=560,
    legend=dict(traceorder="reversed", font=dict(size=14), x=1.02, y=1),
    title=dict(font=dict(size=24), x=0.5)
)

# Axes
fig.update_xaxes(
    tickvals=YEARS_TICKS,
    ticktext=[str(y) for y in YEARS_TICKS],
    tickangle=45,
    title_font=dict(size=16),
    tickfont=dict(size=14),
)

# Only set title on the left y-axis
fig.update_yaxes(
    title=Y_AXIS_TITLE,
    showgrid=True, gridcolor="lightgray",
    title_font=dict(size=16),
    tickfont=dict(size=14),
    rangemode="tozero",
    row=1, col=1
)

# Keep right y-axis without a title
fig.update_yaxes(
    showgrid=True, gridcolor="lightgray",
    tickfont=dict(size=14),
    rangemode="tozero",
    row=1, col=2
)

# Bump subplot title size
fig.update_annotations(font=dict(size=18))

# Save & show
out_path = "./fig/bld_energy_type.png"
pio.write_image(fig, out_path, width=900, height=560, scale=2)  # requires kaleido
fig
